In [31]:
import numpy as np
import math
import matplotlib.pyplot as plt
from numba import njit
import scipy
from itertools import product


In [32]:
@njit
def QUBO_energy(x, Q):
    return x @ Q @ x

@njit
def sweep_once(x, Q, T, E):
    N = len(x)
    for _ in range(N):
        i = np.random.randint(N) 

        old_bit = x[i]
        x[i] = 1 - x[i]  # Flip the binary spin (0 to 1 or 1 to 0)
        
        # Calculate the change in energy if we flip spin i
        new_E = QUBO_energy(x, Q)
        delta_E = new_E - E
        
        # Metropolis criterion
        if delta_E < 0 or np.random.rand() < np.exp(-delta_E / T):
            E = new_E  # Accept the new configuration
        else:
            x[i] = old_bit  # Revert the flip if not accepted  
            
    return E

def sim_annealing(Q, T_start, T_end, alpha, sweeps_per_T):
    n = Q.shape[0]
    x = np.random.choice(np.array([0, 1]), size=n).astype(np.float64)

    E = QUBO_energy(x, Q)

    temps = []
    energies = []
    
    T = T_start
    
    while T > T_end:
        for _ in range(sweeps_per_T):
            E = sweep_once(x, Q, T, E)
        
        temps.append(T)
        energies.append(E)
        T *= alpha
    
    return np.array(temps), np.array(energies), x.copy()


In [33]:
def brute_force_qubo(Q):
    n = Q.shape[0]

    best_energy = np.inf # set  to infiniy  
    best_x = None

    for x in product([0, 1], repeat=n): #produce generate all possible binary configurations of length n
        x = np.array(x, dtype=np.int8)
        E = x @ Q @ x

        if E < best_energy:
            best_energy = E
            best_x = x.copy()

    return best_energy, best_x

In [34]:
Q = np.array([
    [1, -10, -99],
    [ 0, -9, -99],
    [ 0,  0, -99]
], dtype=np.float64)

E_exact, x_exact = brute_force_qubo(Q)

print(E_exact)
print(x_exact)

-315.0
[1 1 1]


In [35]:
temps, energies, x_sa = sim_annealing(
    Q,
    T_start=10,
    T_end=0.01,
    alpha=0.95,
    sweeps_per_T=100
)

print("SA energy:", QUBO_energy(x_sa, Q))
print("SA solution:", x_sa)

print("Exact energy:", E_exact)
print("Exact solution:", x_exact)

SA energy: -315.0
SA solution: [1. 1. 1.]
Exact energy: -315.0
Exact solution: [1 1 1]
